# Trakt API  
![alt text](images/trakt_logo.jpg)   
  
https://trakt.tv   
Trakt is web-based service that automatically tracks, syncs, and organizes the TV shows and movies you watch across various streaming services and media centers. It helps users discover new content, manage watchlists, and share ratings, acting as a centralized hub for managing your media consumption history over time.  
I have been using Trakt for several years to manage my watch history, and I have an extensive history of using this service. **I can use their API to extract my personal media watch history.**

Details of the Trakt APi is here ...  
https://trakt.docs.apiary.io/#introduction/create-an-app  

The Trakt API uses **OAuth** for Authentication and there are a few steps to get setup  
https://trakt.docs.apiary.io/#reference/authentication-oauth  

## Creating an App for Trakt API  
The Trakt API needs to use OAuth to grant users access its data.   
I initially setup a Trakt API App on https://trakt.tv/oauth/applications/new  
  
This a API App has an associated 
* API Client ID
* API Client Secret    
(saved locally)

### One-Time PIN

I then used https://trakt.tv/oauth/authorize to get one-time PIN with parameters response_type, client_id, redirect_uri, scope

<img src="images/trakt_auth.png" alt="trakt" width="300">


### OAuth Token
This One Time PIN is then used in the PostMan call to (POST) https://api.trakt.tv/oauth/token
with the following JSON in POST raw Body to get OAuth Token:  
<font color="lightblue">{  
  "code": XXXXXX,  
  "client_id": XXXXXXX,  
  "client_secret": XXXXXXXXX,  
  "redirect_uri": "urn:ietf:wg:oauth:2.0:oob",  
  "grant_type": "authorization_code"  
}</font>  
  
This will return the OAuth Token  
  
<font color="lightblue">{  
    "access_token": "XXXXXXXXXXXXXXXXXXXXXXXX",  
    "token_type": "Bearer",  
    "expires_in": XXXXX,  
    "refresh_token": "XXXXXXXXXXXXXXXXXXXXXXXXX",  
    "scope": "public",  
    "created_at": 1773755913  
}</font>  
  
You should not need to regenerate a PIN ... but you can regenerate a Token with another call to https://api.trakt.tv/oauth/token  
Again with the following in the Body raw json  

<font color="lightblue">{  
  "refresh_token": "XXXXXXXXXXXXXXXXXXXXXXXXXX",  
  "client_id": [API Client ID],  
  "client_secret": [API Client Secret],  
  "redirect_uri": "urn:ietf:wg:oauth:2.0:oob",  
  "grant_type": "refresh_token"  
}</font>  

***
## Extracting Watch History with API 
The Trakt API call endpoints returns details on my watched history.  
* /sync/history   
* /sync/watched/shows/  
* /sync/watchlist  
* /sync/me  

I can also extract the rating I submit to Trakt for my watched episodes .
This is a separate Endpoint
* /sync/ratings/episodes

  
I used Postman to test these endpoints extensively


The <font color="crimson">/sync/watched/shows</font> endpoint returns aggregated data (playcounts), not the raw watch events  
The JSON returned by this end point has the following structure (for example, the show "A Thousand Blows", where I watched 3 episodes):  

The "rating" endpoint returns the following (sample) json data for the rating of episode 5 of show "A Thousand Blows"

I intend to use the Trakt API endpoints
* /sync/watched/shows/
* /sync/ratings/episodes

... to extract the following data items :
1. "tmdb_id"
1. "title"
1. "season_number"
1. "season_episode_number"
1. "last_watched_at"
1. "rated_at"
1. "rating"

***
### Calling API with Python  
Below is an example of "test" python code that calls these 2 API endpoints to extract a json-based data records of my watch history and rating history and merges collected data into a simpler dataframe.  
1. The code extracts the fields listed above from 2 different endpoints.  
2. Data extracted from each endpoint is saved into dataframes with one row per episode  
3. The 2 dataframes are then merged.  

In [3]:
import requests
import config as cfg # Configuration Details for Trakt API
import pandas as pd

import importlib
importlib.reload(cfg) #Annoying Notebook Caching Issues

# This file is for testing the Trakt API connection and tokens. It will print the name/id/watcheddate of shows in my watched history.
headers = {
    "Authorization": f"Bearer {cfg.TRAKT_TOKENS['trakt_access_token']}",
    "trakt-api-key": cfg.TRAKT_TOKENS['trakt_client_id'],
    "trakt-api-version": "2",
    "Content-Type": "application/json"
}


url = cfg.TRAKT_TOKENS['trakt_url']
# I will use the /sync/watched/shows/ endpoint to get my watched history. 
watched_ep = "/sync/watched/shows/"
r = requests.get(url + watched_ep, headers=headers)
watched_data = r.json()

# I will use the /sync/ratings/episodes endpoint to get my episode ratings. 
ratings_ep = "/sync/ratings/episodes"
r = requests.get(url + ratings_ep, headers=headers)
ratings_data = r.json()

watched_rows = []
# Flatten the watched data to get a row for each episode
# with the show title/id, season number, episode number, and last watched date.
for show_item in watched_data:
    show = show_item.get("show", {})
    show_ids = show.get("ids", {})
    tmdb_id = show_ids.get("tmdb")
    title = show.get("title")

    for season in show_item.get("seasons", []):
        season_number = season.get("number")

        for episode in season.get("episodes", []):
            watched_rows.append({
                "tmdb_id": tmdb_id,
                "title": title,
                "season_number": season_number,
                "season_episode_number": episode.get("number"),
                "last_watched_at": episode.get("last_watched_at")
            })

watched_df = pd.DataFrame(watched_rows)

ratings_rows = []
# Flatten the ratings data to get a row for each episode rating 
# with the show id, season number, episode number, rating, and rated date.
for rating_item in ratings_data:
    show = rating_item.get("show", {})
    show_ids = show.get("ids", {})
    episode = rating_item.get("episode", {})

    ratings_rows.append({
        "tmdb_id": show_ids.get("tmdb"),
        "season_number": episode.get("season"),
        "season_episode_number": episode.get("number"),
        "rated_at": rating_item.get("rated_at"),
        "rating": rating_item.get("rating")
    })

ratings_df = pd.DataFrame(ratings_rows)

final_df = watched_df.merge(
    ratings_df,
    on=["tmdb_id", "season_number", "season_episode_number"],
    how="left" # use left join as not all watched episodes will have ratings
)

print(final_df)

      tmdb_id               title  season_number  season_episode_number  \
0       97951  Mayor of Kingstown              1                      1   
1       97951  Mayor of Kingstown              1                      2   
2       97951  Mayor of Kingstown              1                      3   
3       97951  Mayor of Kingstown              1                      4   
4       97951  Mayor of Kingstown              1                      5   
...       ...                 ...            ...                    ...   
3468     1396        Breaking Bad              5                     12   
3469     1396        Breaking Bad              5                     13   
3470     1396        Breaking Bad              5                     14   
3471     1396        Breaking Bad              5                     15   
3472     1396        Breaking Bad              5                     16   

               last_watched_at                  rated_at  rating  
0     2022-04-14T23:29:00.000Z  

This was a trial version of the final python code for collecting trakt data.  
The final version was reconfigured for separating the RESTful API calls from the core "service" code.

The final code version (within project) is here within these 2 files :  
[trakt_manager.py](trakt_manager.py)  
[api/api_trakt.py](api/api_trakt.py)  

##### <font color="orange">This list of watched TV Shows will act as the initial baseline collection of Shows that will be added to the new TV Show Database.</font>  
The full details of these TV Shows (seasons/episodes/cast/etc.) will be extracted from the TMDB API Service.